<a href="https://colab.research.google.com/github/ovifernandez/pruebaopengeoai/blob/develop/model-trainer-kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
# 1. Instalamos geoai (al ser muy ligero, dejamos que lo baje de internet)
%pip install geoai-py -q


In [4]:
import os
import geoai
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [ ]:
train_raster_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B7/B7.tif"
train_vector_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B7/GroundTruth_B7.geojson"
test_raster_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B9/20220714_FLEXIGROBOTS_B9_CIR.tif"


In [ ]:
geoai.get_raster_info(train_raster_path)

{'driver': 'GTiff',
 'width': 11804,
 'height': 14284,
 'count': 5,
 'dtype': 'float32',
 'crs': 'EPSG:32629',
 'transform': Affine(0.013460000000000003, 0.0, 517084.75678000005,
        0.0, -0.013460000000000003, 4645195.688750001),
 'bounds': BoundingBox(left=517084.75678000005, bottom=4645003.42611, right=517243.63862000004, top=4645195.688750001),
 'resolution': (0.01, 0.01),
 'nodata': -10000.0,
 'band_stats': [{'band': 1,
   'min': 0.006998511962592602,
   'max': 0.44269442558288574,
   'mean': 0.05499494534554005,
   'std': 0.017451611113407484},
  {'band': 2,
   'min': 0.017103200778365135,
   'max': 0.5450941324234009,
   'mean': 0.1045741604533924,
   'std': 0.026651857959112027},
  {'band': 3,
   'min': 0.01057412289083004,
   'max': 0.6046684384346008,
   'mean': 0.13974507614974113,
   'std': 0.05278399517164988},
  {'band': 4,
   'min': 0.03274083137512207,
   'max': 0.707343339920044,
   'mean': 0.20943387272359748,
   'std': 0.04619268717657372},
  {'band': 5,
   'min'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
style_dict = {
    "color": "#ff0000",
    "weight": 2,
    "opacity": 1,
    # "fill": True,
    # "fillColor": "#ffffff",
    "fillOpacity": 0,
    # "dashArray": "9"
    # "clickable": True,
}
style_function = lambda x: style_dict

geoai.view_vector_interactive(
    train_vector_path, tiles=train_raster_path, style_function=style_function
)

In [ ]:
geoai.view_raster(test_raster_path)

In [9]:
import shutil
import os

ruta_drive_zip = "/content/drive/MyDrive/AGRIA/input/parcelas/B7/dataset_B7_CIR.zip"
ruta_local_zip = "/content/dataset_B7.zip"
input_folder = "/content/dataset_local"

print("1. Copiando el bloque de 10GB desde Drive al SSD local (Paciencia, tardará unos minutos pero será continuo)...")
shutil.copy(ruta_drive_zip, ruta_local_zip)

print("2. Descomprimiendo archivos en el SSD...")
os.makedirs(input_folder, exist_ok=True)
!unzip -q {ruta_local_zip} -d {input_folder}

print("3. Limpiando el zip local para liberar espacio...")
os.remove(ruta_local_zip)
print("¡Listo! I/O optimizado al máximo.")

1. Copiando el bloque de 10GB desde Drive al SSD local (Paciencia, tardará unos minutos pero será continuo)...
2. Descomprimiendo archivos en el SSD...
3. Limpiando el zip local para liberar espacio...
¡Listo! I/O optimizado al máximo.


In [10]:
input_folder_local = "/content/dataset_local/B7_CIR"
os.makedirs(input_folder_local, exist_ok=True)
out_folder = "/content/out/B7_CIR"
os.makedirs(out_folder, exist_ok=True)

Entrenamos el modelo Mask R-CNN sobre nuestros tiles generados, para que clasifique, localice bboxes y aplique máscaras a cada cepa detectada.

In [11]:
geoai.train_instance_segmentation_model(
    images_dir=f"{input_folder}/images",
    labels_dir=f"{input_folder}/labels",
    output_dir=f"{out_folder}/instance_models",
    num_classes=2,  # clase fondo y clase cepa. En un futuro, se añadirá clase tronco
    num_channels=3, # 3 para imágenes RGB, 5 para imágenes MSP
    batch_size=4, # Para no consumir excesiva VRAM, y no provocar un error de Out of Memory a mitad de ejecución.
    num_epochs=30, # 10 para una PoC, 50 para entrenamiento real con dataset augmentado.
    learning_rate=0.0005, # Learning rate menos agresivo que el de por defecto, para un descenso de gradiente suave y controlado con un batch sizze de 4.
    val_split=0.2,
    visualize=False,
    verbose=True,
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_local/images (1)'

In [ ]:
# Definimos las rutas a las máscaras predichas
masks_path = f"{out_folder}/test_instance_prediction.tif"
model_path = f"{out_folder}/instance_models/best_model.pth"

In [ ]:
geoai.view_raster(test_raster_path)

In [ ]:
# Inferencia sobre nuevo ortomosaico, de la parcela B9
geoai.instance_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=512,
    overlap=256,
    confidence_threshold=0.6,
    batch_size=4,
)

Mounted at /content/drive


In [ ]:
masks_path_high_conf = f"{out_folder}/test_instance_prediction_high_conf.tif"


In [ ]:
geoai.instance_segmentation(
    input_path=test_raster_path,
    output_path=masks_path_high_conf,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=512,
    overlap=256,
    confidence_threshold=0.7,  # Higher threshold for more confident predictions
    batch_size=4,
)

In [ ]:
output_vector_path = "test_instance_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_vector_path, epsilon=2)